# 055 — Atención y arquitectura Transformer

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución explicada

**Ejercicio 1.** Q·Kᵀ = [[2,0],[0,2]]; escaladas: [[1.4142, 0], [0, 1.4142]].
Softmax fila 1: (4.1133, 1)/5.1133 = **(0.8044, 0.1956)** (fila 2 simétrica).
Salida fila 1: 0.8044·(1,0) + 0.1956·(0,2) = **(0.8044, 0.3912)**;
fila 2: 0.1956·(1,0) + 0.8044·(0,2) = **(0.1956, 1.6088)**.

**Ejercicio 2.** Sin escalar, fila 1 = softmax(2, 0) = (e², 1)/(e²+1) =
**(0.8808, 0.1192)**: más concentrada que (0.8044, 0.1956). Con d_k = 512 las
puntuaciones crecen ~√d_k y el softmax se vuelve casi one-hot: gradientes ≈ 0 hacia
los tokens no ganadores — por eso el escalado es parte de la definición.

**Ejercicio 3.** La fila 1 enmascara la posición 2 (−∞): softmax = (1, 0) → salida =
V₁ = **(1, 0)**. La fila 2 no cambia: **(0.1956, 1.6088)**. La posición 1 solo puede
"copiarse a sí misma": así se preserva la causalidad del generador.

**Ejercicio 4.** Atención: 4·512·512 = **1 048 576**. FFN: 512·2048 + 2048·512 =
**2 097 152**. La FFN dobla a la atención en parámetros — patrón general en
Transformers (de ahí que las técnicas de eficiencia ataquen ambas partes).


In [ ]:
result = run_lab("attention", seed=55)
assert result["kind"] == "attention"
assert result["evidence"]
show(result)


In [ ]:
# Verificación numérica
import math

def softmax(row):
    m = max(row)
    e = [math.exp(v - m) for v in row]
    s = sum(e)
    return [v / s for v in e]

Q = [[1, 0], [0, 1]]
K = [[2, 0], [0, 2]]
V = [[1, 0], [0, 2]]
dk = 2

scores = [[sum(Q[i][k] * K[j][k] for k in range(2)) / math.sqrt(dk)
           for j in range(2)] for i in range(2)]
A = [softmax(row) for row in scores]
out = [[sum(A[i][j] * V[j][c] for j in range(2)) for c in range(2)] for i in range(2)]
print("A =", [[round(v, 4) for v in r] for r in A])
print("out =", [[round(v, 4) for v in r] for r in out])
assert abs(A[0][0] - 0.8044) < 1e-3 and abs(out[0][0] - 0.8044) < 1e-3

# Ejercicio 3: máscara causal
masked = [[scores[0][0], float("-inf")], scores[1]]
A_causal = [softmax(r) for r in masked]
out_causal = [[sum(A_causal[i][j] * V[j][c] for j in range(2)) for c in range(2)]
              for i in range(2)]
print("out causal =", [[round(v, 4) for v in r] for r in out_causal])
assert out_causal[0] == [1.0, 0.0]

# Ejercicio 4
print("atención:", 4 * 512 * 512, "| FFN:", 512 * 2048 + 2048 * 512)


## Reflexión

1. ¿Qué problema concreto de la LSTM resuelve que el camino entre dos posiciones cualesquiera sea O(1), y qué precio computacional se paga (O(n²))?
2. Si eliminas la codificación posicional de un Transformer entrenado, ¿qué tareas seguirían funcionando y cuáles colapsarían?
3. ¿Por qué la máscara causal permite entrenar en paralelo un modelo que en inferencia es estrictamente secuencial?
